In [2]:
import os
import numpy as np
from collections import defaultdict
import matplotlib.pyplot as plt
import cv2
import random
import shutil
from PIL import Image

In [3]:
train_images_loc = r"/home/faustino/Documents/detection_project/yolo_ensemble/datasets/all/images/train"
train_labels_loc  = r"/home/faustino/Documents/detection_project/yolo_ensemble/datasets/all/labels/train"

In [7]:
def get_labels(path):
    labels = []
    for root, _, files in os.walk(path):
        for file in files:
            file_path = os.path.join(root, file)
            L = []
            with open(file_path, 'r') as f:
                for line in f:
                    parts = line.split()
                    if parts:
                        L.append(int(parts[0]))
            labels.append(L)
    return labels

y_train_labels = get_labels(train_labels_loc)

In [11]:
frequent_classes = [1, 2, 6, 10]
rare_classes = [0, 3, 4, 5, 7, 8, 9, 11, 12, 13]
nb=len(y_train_labels)
print(nb)

1193


In [ ]:
for image_labels in y_train_labels:
    for label in image_labels:
        if label in frequent_classes:
            write_to_frequent()
        if labelin rare_classes:
            write_to_rare()

In [12]:
import os
import shutil

def split_yolo_dataset(
    images_dir=r"/home/faustino/Documents/detection_project/yolo_ensemble/datasets/all/images/train", 
    labels_dir=r"/home/faustino/Documents/detection_project/yolo_ensemble/datasets/all/labels/train",
    frequent_classes=frequent_classes, rare_classes=rare_classes, copy_images=True):
    
    """
    Split YOLO dataset into frequent / rare subsets by filtering bounding boxes.
    """
    
    freq_img_dir = r"/home/faustino/Documents/detection_project/yolo_ensemble/datasets/frequent/images"
    freq_lbl_dir = r"/home/faustino/Documents/detection_project/yolo_ensemble/datasets/frequent/labels"
    rare_img_dir = r"/home/faustino/Documents/detection_project/yolo_ensemble/datasets/rare/images"
    rare_lbl_dir = r"/home/faustino/Documents/detection_project/yolo_ensemble/datasets/rare/labels"

    os.makedirs(freq_img_dir, exist_ok=True)
    os.makedirs(freq_lbl_dir, exist_ok=True)
    os.makedirs(rare_img_dir, exist_ok=True)
    os.makedirs(rare_lbl_dir, exist_ok=True)

    for label_file in os.listdir(labels_dir):
        if not label_file.endswith(".txt"):
            continue

        label_path = os.path.join(labels_dir, label_file)
        image_name = os.path.splitext(label_file)[0]

        # image extension handling (.jpg / .png)
        image_path = None
        for ext in [".jpg", ".png", ".jpeg"]:
            candidate = os.path.join(images_dir, image_name + ext)
            if os.path.exists(candidate):
                image_path = candidate
                break

        if image_path is None:
            print(f"[WARN] Image not found for {label_file}")
            continue

        frequent_lines = []
        rare_lines = []

        with open(label_path, "r") as f:
            for line in f:
                class_id = int(line.split()[0])

                if class_id in frequent_classes:
                    frequent_lines.append(line)
                elif class_id in rare_classes:
                    rare_lines.append(line)

        # Write frequent labels + image
        if frequent_lines:
            out_label = os.path.join(freq_lbl_dir, label_file)
            with open(out_label, "w") as f:
                f.writelines(frequent_lines)

            out_image = os.path.join(freq_img_dir, os.path.basename(image_path))
            _copy_or_link(image_path, out_image, copy_images)

        # Write rare labels + image
        if rare_lines:
            out_label = os.path.join(rare_lbl_dir, label_file)
            with open(out_label, "w") as f:
                f.writelines(rare_lines)

            out_image = os.path.join(rare_img_dir, os.path.basename(image_path))
            _copy_or_link(image_path, out_image, copy_images)


def _copy_or_link(src, dst, copy_images):
    if os.path.exists(dst):
        return
    if copy_images:
        shutil.copy2(src, dst)
    else:
        os.symlink(os.path.abspath(src), dst)

In [13]:
split_yolo_dataset()

In [14]:
def modify_class_labels(labels_path, class_mapping):
    """
    Renumber YOLO class labels according to a mapping.
    """

    for root, _, files in os.walk(labels_path):
        for file in files:
            if not file.endswith(".txt"):
                continue

            file_path = os.path.join(root, file)
            new_lines = []

            with open(file_path, "r") as f:
                for line in f:
                    parts = line.strip().split()
                    if not parts:
                        continue

                    old_label = int(parts[0])

                    if old_label not in class_mapping:
                        # Skip labels not in this dataset
                        continue

                    new_label = class_mapping[old_label]
                    parts[0] = str(new_label)
                    new_lines.append(" ".join(parts) + "\n")

            # Overwrite file only if there are valid labels
            if new_lines:
                with open(file_path, "w") as f:
                    f.writelines(new_lines)
            else:
                # Remove empty label files
                os.remove(file_path)

In [16]:
frequent_mapping = {
    1: 0,   # Building
    2: 1,   # Crosswalk
    6: 2,   # Medium Vehicle
    10: 3   # Small Vehicle
}

modify_class_labels(r"/home/faustino/Documents/detection_project/yolo_ensemble/datasets/frequent/labels", frequent_mapping)

In [17]:
rare_mapping = {
    0: 0,    # Basketball Field
    3: 1,    # Football Field
    4: 2,    # Graveyard
    5: 3,    # Large Vehicle
    7: 4,    # Playground
    8: 5,    # Roundabout
    9: 6,    # Ship
    11: 7,   # Swimming Pool
    12: 8,   # Tennis Court
    13: 9    # Train
}

modify_class_labels(r"/home/faustino/Documents/detection_project/yolo_ensemble/datasets/rare/labels", rare_mapping)